In [3]:
import pandas as pd
import icartt
import os
import warnings
import re
from datetime import datetime
import csv
from datetime import datetime, timedelta
from netCDF4 import Dataset
import numpy as np
from scipy import stats
import glob
from math import pi
import ast

In [4]:
import pandas as pd
import numpy as np

# Define organizations and paths
ORGANIZATIONS = ["NASA", "DOE", "NSF", "NOAA"]

NSF_RESTRICTED_PATH = rf"C:\Users\haika\Desktop\May_Research\may_datasets\restricted_combined\NSF_restricted.csv"
NOAA_RESTRICTED_PATH = rf"C:\Users\haika\Desktop\May_Research\may_datasets\restricted_combined\NOAA_restricted.csv"
NASA_RESTRICTED_PATH = rf"C:\Users\haika\Desktop\May_Research\may_datasets\restricted_combined\NASA_restricted.csv"
DOE_RESTRICTED_PATH = rf"C:\Users\haika\Desktop\May_Research\may_datasets\restricted_combined\DOE_restricted.csv"

NSF_COMPREHENSIVE_PATH = rf"C:\Users\haika\Desktop\May_Research\may_datasets\comprehensive_combined\NSF_comprehensive.csv"
NOAA_COMPREHENSIVE_PATH = rf"C:\Users\haika\Desktop\May_Research\may_datasets\comprehensive_combined\NOAA_comprehensive.csv"
NASA_COMPREHENSIVE_PATH = rf"C:\Users\haika\Desktop\May_Research\may_datasets\comprehensive_combined\NASA_comprehensive.csv"
DOE_COMPREHENSIVE_PATH = rf"C:\Users\haika\Desktop\May_Research\may_datasets\comprehensive_combined\DOE_comprehensive.csv"

# Dictionary mapping organizations to their paths
paths_dict = {
    "NSF": {
        "restricted": NSF_RESTRICTED_PATH,
        "comprehensive": NSF_COMPREHENSIVE_PATH
    },
    "NOAA": {
        "restricted": NOAA_RESTRICTED_PATH,
        "comprehensive": NOAA_COMPREHENSIVE_PATH
    },
    "NASA": {
        "restricted": NASA_RESTRICTED_PATH,
        "comprehensive": NASA_COMPREHENSIVE_PATH
    },
    "DOE": {
        "restricted": DOE_RESTRICTED_PATH,
        "comprehensive": DOE_COMPREHENSIVE_PATH
    }
}

def estimate_file_size_mb(df):
    """
    Estimate the file size of a DataFrame in MB when saved as CSV.
    This is an approximation based on memory usage.
    """
    # Get memory usage in bytes
    memory_usage = df.memory_usage(deep=True).sum()
    
    # Adjusted multiplier based on actual FIREXAQ data:
    # 717.32 MB estimated vs 176 MB actual = 4.07x overestimate
    # New multiplier: 1.8 / 4.07 ≈ 0.44
    estimated_csv_size = memory_usage * 0.44
    
    # Convert to MB
    size_mb = estimated_csv_size / (1024 * 1024)
    return size_mb

def analyze_campaign_data(file_path, org_name, dataset_type):
    """
    Analyze campaign data from a given file path.
    Returns a summary of campaigns with row counts and estimated file sizes.
    """
    try:
        print(f"\nAnalyzing {org_name} {dataset_type.upper()} dataset...")
        print(f"File: {file_path}")
        
        # Read the CSV file
        df = pd.read_csv(file_path)
        
        print(f"Total rows: {len(df):,}")
        print(f"Total columns: {len(df.columns)}")
        print(f"Total estimated file size: {estimate_file_size_mb(df):.2f} MB")
        
        # Check if Campaign column exists
        if 'Campaign' not in df.columns:
            print("WARNING: 'Campaign' column not found!")
            print(f"Available columns: {list(df.columns)}")
            return None
        
        # Group by Campaign and analyze
        campaign_analysis = df.groupby('Campaign').agg({
            'Campaign': 'count'  # Count rows per campaign
        }).rename(columns={'Campaign': 'Row_Count'})
        
        # Calculate estimated file size per campaign
        campaign_sizes = []
        for campaign in campaign_analysis.index:
            campaign_df = df[df['Campaign'] == campaign]
            estimated_size = estimate_file_size_mb(campaign_df)
            campaign_sizes.append(estimated_size)
        
        campaign_analysis['Estimated_Size_MB'] = campaign_sizes
        
        # Sort by row count (descending)
        campaign_analysis = campaign_analysis.sort_values('Row_Count', ascending=False)
        
        print(f"\nCampaign breakdown for {org_name} {dataset_type.upper()}:")
        print("=" * 60)
        print(f"{'Campaign':<25} {'Rows':<12} {'Est. Size (MB)':<15}")
        print("-" * 60)
        
        total_rows = 0
        total_size = 0
        
        for campaign, row in campaign_analysis.iterrows():
            rows = int(row['Row_Count'])
            size_mb = row['Estimated_Size_MB']
            total_rows += rows
            total_size += size_mb
            print(f"{campaign:<25} {rows:<12,} {size_mb:<15.2f}")
        
        print("-" * 60)
        print(f"{'TOTAL':<25} {total_rows:<12,} {total_size:<15.2f}")
        
        return campaign_analysis
        
    except FileNotFoundError:
        print(f"ERROR: File not found - {file_path}")
        return None
    except Exception as e:
        print(f"ERROR processing {file_path}: {str(e)}")
        return None

# Main analysis
print("CAMPAIGN DATA ANALYSIS")
print("=" * 80)

# Store results for summary
all_results = {}

# Analyze each organization's datasets
for org in ORGANIZATIONS:
    all_results[org] = {}
    
    # Analyze restricted dataset
    restricted_result = analyze_campaign_data(
        paths_dict[org]["restricted"], 
        org, 
        "restricted"
    )
    all_results[org]["restricted"] = restricted_result
    
    # Analyze comprehensive dataset
    comprehensive_result = analyze_campaign_data(
        paths_dict[org]["comprehensive"], 
        org, 
        "comprehensive"
    )
    all_results[org]["comprehensive"] = comprehensive_result

# Summary across all organizations
print("\n" + "=" * 80)
print("SUMMARY ACROSS ALL ORGANIZATIONS")
print("=" * 80)

for dataset_type in ["restricted", "comprehensive"]:
    print(f"\n{dataset_type.upper()} DATASETS:")
    print("-" * 40)
    
    for org in ORGANIZATIONS:
        result = all_results[org][dataset_type]
        if result is not None:
            total_rows = result['Row_Count'].sum()
            total_size = result['Estimated_Size_MB'].sum()
            num_campaigns = len(result)
            print(f"{org:<6}: {num_campaigns:>2} campaigns, {total_rows:>8,} rows, {total_size:>6.1f} MB")
        else:
            print(f"{org:<6}: No data available")

CAMPAIGN DATA ANALYSIS

Analyzing NASA RESTRICTED dataset...
File: C:\Users\haika\Desktop\May_Research\may_datasets\restricted_combined\NASA_restricted.csv
Total rows: 4,710,510
Total columns: 32
Total estimated file size: 725.72 MB

Campaign breakdown for NASA RESTRICTED:
Campaign                  Rows         Est. Size (MB) 
------------------------------------------------------------
FIREXAQ                   1,119,510    175.22         
SEAC4RS                   605,493      94.77          
CAMP2Ex                   530,889      83.09          
ASIA-AQ                   437,326      68.45          
DC3                       408,424      63.24          
DISCOVERAQ-DC             364,225      57.92          
NAAMES(2016)              310,775      49.29          
DISCOVERAQ-California     280,646      45.57          
NAAMES(2017)              278,797      44.22          
DISCOVERAQ-Texas          208,585      33.43          
NAAMES(2015)              165,840      26.30          
-----